# Four targets: analytic integration convergence

本 notebook 只读取现有 `validation.npz`，不训练、不加载模型权重。
目的：区分“积分尚不稳定”和“解析多路线场稳定地漏点”。

**运行：** 放在原 `streaming-flow-policy` 项目中，按顺序 Run All。
找不到数据时，只需在下一格设置 `PROJECT_ROOT_OVERRIDE`。
使用原 Python 环境；若缺包，在该环境安装 `numpy pandas scipy matplotlib`。
修改配置后请 Restart Kernel，再 Run All，以免复用旧的内存变量。

默认先检查布局 12、20、281、954，再复核原来的分层随机样本
（每组 25 个，selection seed = 2026）。编号均为 **从 0 开始**。
固定物理起点；场内的 Gaussian 宽度仍为 `0.05 * exp(-2.5*t)`。
这里没有把场内宽度设为零。

输出到新的时间戳目录，不覆盖训练数据、模型或旧诊断结果。
日志和图使用英文，说明使用中文。本文件交付前只做静态检查，未执行实验。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import itertools
import json
import os
import platform
import time
import uuid

try:
    import numpy as np
    import pandas as pd
    import scipy
    from scipy.integrate import solve_ivp
    import matplotlib.pyplot as plt
    from IPython.display import display
except ImportError as exc:
    raise ImportError(
        "Missing dependency. In this notebook's Python environment run: "
        "python -m pip install numpy pandas scipy matplotlib ipython"
    ) from exc

PROJECT_ROOT_OVERRIDE = None  # e.g. Path(r"D:/projects/streaming-flow-policy")
RUN_FULL_SAMPLE = True        # False = only the four pilot layouts
PILOT_LAYOUTS = [12, 20, 281, 954]
N_PER_GAP_BIN = 25
LAYOUT_SELECTION_SEED = 2026

# These are the constants in your original training notebook.
SIGMA_0 = 0.05
GAIN = 2.5
RELATIVE_SLACK = 0.05
VISIT_TOLERANCES = (0.02, 0.05)
RK4_LEVELS = (1000, 2000, 4000, 8000)
DOP_SETTINGS = {
    "dop853": dict(rtol=1e-8, atol=1e-10),
    "dop853_tight": dict(rtol=1e-10, atol=1e-12),
}
DOP_MAX_STEP = 0.005

# Practical agreement criteria, not rigorous global error bounds.
POSITION_TOL = 0.001
DISTANCE_TOL = 0.0002
BOUNDARY_MARGIN = 0.001
SINGLE_EXACT_TOL = 1e-6
GAP_LABELS = ["<1%", "1–<5%", "5–<10%", "≥10%"]
METHODS = ("analytic_single", "analytic_multi")

assert SIGMA_0 > 0 and GAIN >= 0
assert len(RK4_LEVELS) == 4 and min(RK4_LEVELS) > 0
assert list(RK4_LEVELS) == sorted(set(RK4_LEVELS))
assert all(b == 2 * a for a, b in zip(RK4_LEVELS[:-1], RK4_LEVELS[1:]))
assert N_PER_GAP_BIN > 0 and POSITION_TOL > 0 and DISTANCE_TOL > 0
assert BOUNDARY_MARGIN >= POSITION_TOL
COMMON_STEPS = max(RK4_LEVELS)
COMMON_TIMES = np.linspace(0.0, 1.0, COMMON_STEPS + 1)
FINE_TIMES = np.linspace(0.0, 1.0, 2 * COMMON_STEPS + 1)

def find_project_root():
    suffix = Path("experiments/four_boxes_experiments/four_targets/datasets/v1/validation.npz")
    if PROJECT_ROOT_OVERRIDE is not None:
        root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not (root / suffix).is_file():
            raise FileNotFoundError(f"Validation file not found: {root / suffix}")
        return root
    here = Path.cwd().resolve()
    for root in [here, *here.parents]:
        if (root / suffix).is_file():
            return root
    raise FileNotFoundError("Set PROJECT_ROOT_OVERRIDE to your streaming-flow-policy repository.")

PROJECT_ROOT = find_project_root()
TASK_ROOT = PROJECT_ROOT / "experiments/four_boxes_experiments/four_targets"
VALIDATION_PATH = TASK_ROOT / "datasets/v1/validation.npz"
print("Validation:", VALIDATION_PATH)
print("Analytic fields only; float64 on CPU; no training or checkpoint loading.")


## 1. Read the same validation data and reconstruct the same experts

`single` 使用原文件里的路线和时间结点；`multi` 枚举 24 种访问顺序，
保留长度不超过 `1.05 * 最短长度` 的全部候选，先验等权。
抽样算法与上一份诊断 notebook 一致；额外的典型案例不混入随机样本总体。


In [ ]:
with np.load(VALIDATION_PATH, allow_pickle=False) as archive:
    required = {"starts", "targets", "routes", "knot_tau", "lengths"}
    missing = required - set(archive.files)
    if missing:
        raise ValueError(f"Validation data missing keys: {sorted(missing)}")
    validation = {name: archive[name].copy() for name in required}

starts = np.asarray(validation["starts"], dtype=np.float64)
targets = np.asarray(validation["targets"], dtype=np.float64)
stored_routes = np.asarray(validation["routes"], dtype=np.float64)
stored_knots = np.asarray(validation["knot_tau"], dtype=np.float64)
N_VALIDATION = len(targets)
for name, array, shape in [
    ("starts", starts, (N_VALIDATION, 2)),
    ("targets", targets, (N_VALIDATION, 4, 2)),
    ("routes", stored_routes, (N_VALIDATION, 5, 2)),
    ("knot_tau", stored_knots, (N_VALIDATION, 5)),
]:
    if array.shape != shape or not np.isfinite(array).all():
        raise ValueError(f"Invalid {name}: expected finite array with shape {shape}")
if not np.all(np.diff(stored_knots, axis=1) > 0):
    raise ValueError("Expert time knots must be strictly increasing.")
np.testing.assert_allclose(stored_knots[:, 0], 0, atol=1e-12, rtol=0)
np.testing.assert_allclose(stored_knots[:, -1], 1, atol=1e-12, rtol=0)
np.testing.assert_allclose(stored_routes[:, 0], starts, atol=1e-12, rtol=0)
if not all(0 <= i < N_VALIDATION for i in PILOT_LAYOUTS):
    raise ValueError("A pilot layout index is outside this validation dataset.")

permutations = np.asarray(list(itertools.permutations(range(4))), dtype=np.int64)
all_routes = np.empty((N_VALIDATION, 24, 5, 2), dtype=np.float64)
for j, order in enumerate(permutations):
    all_routes[:, j] = np.concatenate([starts[:, None], targets[:, order]], axis=1)
segment_lengths = np.linalg.norm(np.diff(all_routes, axis=2), axis=3)
if np.any(segment_lengths <= 0):
    raise ValueError("Zero-length expert segment: inspect the source layouts.")
all_lengths = segment_lengths.sum(axis=2)
rankings = np.argsort(all_lengths, axis=1, kind="stable")
optimal_lengths = all_lengths[np.arange(N_VALIDATION), rankings[:, 0]]
second_lengths = all_lengths[np.arange(N_VALIDATION), rankings[:, 1]]
gap_percent = 100 * (second_lengths - optimal_lengths) / optimal_lengths
layout_gap_bins = np.asarray(GAP_LABELS)[np.searchsorted([1.0, 5.0, 10.0], gap_percent, side="right")]
np.testing.assert_allclose(np.asarray(validation["lengths"]).reshape(-1), optimal_lengths,
                           rtol=2e-6, atol=1e-7)

candidate_ids = {}
for i in range(N_VALIDATION):
    ordered = rankings[i]
    candidate_ids[i] = ordered[
        all_lengths[i, ordered] <= (1 + RELATIVE_SLACK) * optimal_lengths[i] + 1e-12
    ]
    d = np.linalg.norm(stored_routes[i, 1:, None] - targets[i, None], axis=-1)
    order = np.argmin(d, axis=1)
    if not np.array_equal(np.sort(order), np.arange(4)) or np.max(d[np.arange(4), order]) > 1e-7:
        raise ValueError(f"Stored single route does not visit every target: layout {i}")

rng = np.random.default_rng(LAYOUT_SELECTION_SEED)
sample_list, selection_rows = [], []
for label in GAP_LABELS:
    population = np.flatnonzero(layout_gap_bins == label)
    take = min(N_PER_GAP_BIN, len(population))
    picked = rng.choice(population, size=take, replace=False)
    sample_list.extend(picked.tolist())
    selection_rows.append(dict(gap_bin=label, population=len(population), selected=take))
sample_ids = np.asarray(sorted(sample_list), dtype=np.int64)
sample_set = set(sample_ids.tolist())

route_banks = {}
for method in METHODS:
    width = 1 if method == "analytic_single" else max(map(len, candidate_ids.values()))
    vertices = np.empty((N_VALIDATION, width, 5, 2), dtype=np.float64)
    knots = np.empty((N_VALIDATION, width, 5), dtype=np.float64)
    active = np.zeros((N_VALIDATION, width), dtype=bool)
    for i in range(N_VALIDATION):
        if method == "analytic_single":
            vertices[i, 0], knots[i, 0], active[i, 0] = stored_routes[i], stored_knots[i], True
        else:
            ids = candidate_ids[i]
            m = len(ids)
            times = np.concatenate([np.zeros((m, 1)), np.cumsum(segment_lengths[i, ids], axis=1)], axis=1)
            times /= all_lengths[i, ids, None]
            times[:, -1] = 1.0
            vertices[i], knots[i] = all_routes[i, ids[0]], times[0]
            vertices[i, :m], knots[i, :m], active[i, :m] = all_routes[i, ids], times, True
    route_banks[method] = dict(vertices=vertices, knots=knots, active=active)

display(pd.DataFrame(selection_rows))
print("Pilot layouts:", PILOT_LAYOUTS)
print("Stratified sample:", len(sample_ids), "layouts | seed:", LAYOUT_SELECTION_SEED)


## 2. Fresh outputs and exact definitions

对每条专家路线，速度场是 `xi_dot - k * (a - xi)`。
混合权重是当前位置在各条 Gaussian tube 下的后验概率，
**不是一直等权平均速度**。所有方法固定同一起点。

单路线还有精确解：`a(t) = xi(t) + exp(-k*t)*(a(0)-xi(0))`。
本实验中起点与专家相同，因此精确解就是 `xi(t)`；这是求解器的正对照。


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def atomic_json(data, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temp, path)

def atomic_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    os.replace(temp, path)

def atomic_npz(path, **arrays):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    with temp.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temp, path)

run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
OUTPUT_DIR = TASK_ROOT / "diagnostics/analytic_convergence_v2" / run_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
manifest = dict(
    status="started", validation_path=str(VALIDATION_PATH), validation_sha256=sha256_file(VALIDATION_PATH),
    validation_layouts=N_VALIDATION, pilot_layouts=PILOT_LAYOUTS, sample_ids=sample_ids.tolist(),
    sample_seed=LAYOUT_SELECTION_SEED, run_full_sample=RUN_FULL_SAMPLE,
    initial_noise_sigma=0.0, sigma_0=SIGMA_0, gain=GAIN, relative_slack=RELATIVE_SLACK,
    rk4_levels=list(RK4_LEVELS), dop_settings=DOP_SETTINGS, dop_max_step=DOP_MAX_STEP,
    common_grid_intervals=COMMON_STEPS, metric_fine_grid_intervals=2*COMMON_STEPS,
    position_tolerance=POSITION_TOL, distance_tolerance=DISTANCE_TOL,
    boundary_margin=BOUNDARY_MARGIN, single_exact_tolerance=SINGLE_EXACT_TOL,
    python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__,
    pandas=pd.__version__, completed_cases=0,
)
atomic_json(manifest, OUTPUT_DIR / "run_manifest.json")
atomic_csv(pd.DataFrame(selection_rows), OUTPUT_DIR / "sample_selection.csv")
print("Output directory:", OUTPUT_DIR)

def expert_values(vertices, knots, tau):
    segment = np.clip((tau >= knots[..., 1:]).sum(axis=-1), 0, 3)
    b = np.arange(len(vertices))[:, None]
    r = np.arange(vertices.shape[1])[None, :]
    p0, p1 = vertices[b, r, segment], vertices[b, r, segment + 1]
    t0 = knots[b, r, segment]
    dt = knots[b, r, segment + 1] - t0
    derivative = (p1 - p0) / dt[..., None]
    position = p0 + (tau - t0)[..., None] * derivative
    return position, derivative

def make_batch_field(method, indices):
    bank = route_banks[method]
    vertices, knots, active = (bank[key][indices] for key in ("vertices", "knots", "active"))
    def field(actions, tau):
        position, derivative = expert_values(vertices, knots, tau)
        offset = actions[:, None, :] - position
        sigma = SIGMA_0 * np.exp(-GAIN * tau)
        logits = -np.sum(offset * offset, axis=-1) / (2 * sigma * sigma)
        logits = np.where(active, logits, -np.inf)
        logits -= np.max(logits, axis=1, keepdims=True)
        weights = np.exp(logits)
        weights /= weights.sum(axis=1, keepdims=True)
        return np.sum(weights[..., None] * (derivative - GAIN * offset), axis=1)
    return field

def rk4_batch(field, initial, steps):
    # Same unsegmented RK4 arithmetic as the preceding diagnostic notebook.
    actions = np.asarray(initial, dtype=np.float64).copy()
    paths = np.empty((len(actions), steps + 1, 2), dtype=np.float64)
    paths[:, 0] = actions
    dt = 1.0 / steps
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        for step in range(steps):
            tau = step * dt
            k1 = field(actions, tau)
            k2 = field(actions + 0.5 * dt * k1, tau + 0.5 * dt)
            k3 = field(actions + 0.5 * dt * k2, tau + 0.5 * dt)
            k4 = field(actions + dt * k3, min(tau + dt, 1.0))
            actions += dt / 6 * (k1 + 2*k2 + 2*k3 + k4)
            paths[:, step + 1] = actions
    return paths

def exact_single_path(i, times):
    # Linear interpolation at the original time knots; no ODE integration.
    return np.column_stack([
        np.interp(times, stored_knots[i], stored_routes[i, :, d]) for d in range(2)
    ])


## 3. DOP853 split at all expert turning times

每个布局单独求解；断点为该场所有有效专家的时间结点并集。
在一个小区间内，每条专家当前所在的线段是固定的。
在右端点使用该区间的左侧速度极限，下一区间使用右侧极限，
避免求解器在一次步进内跨过速度跳变。实际位置连续传递，不做重置。

同时检查评估精度：用连续插值分别输出 8000 / 16000 个时间区间，
两者都加入转弯结点，然后计算到**轨迹线段**的距离，不只检查采样点。
因而求解器的容限与目标访问判定的采样密度是分别检查的。


In [ ]:
def segmented_dop853(method, i, settings):
    bank = route_banks[method]
    mask = bank["active"][i]
    vertices, knots = bank["vertices"][i, mask], bank["knots"][i, mask]
    boundaries = np.unique(knots.ravel())
    action = starts[i].copy()
    pieces, nfev = [], 0
    r = np.arange(len(vertices))
    for left, right in zip(boundaries[:-1], boundaries[1:]):
        midpoint = left + (right - left) / 2
        segment = np.clip((midpoint >= knots[:, 1:]).sum(axis=1), 0, 3)
        t0 = knots[r, segment]
        p0, p1 = vertices[r, segment], vertices[r, segment + 1]
        derivative = (p1 - p0) / (knots[r, segment + 1] - t0)[:, None]

        def field(tau, a):
            position = p0 + (tau - t0)[:, None] * derivative
            offset = a[None, :] - position
            sigma = SIGMA_0 * np.exp(-GAIN * tau)
            logits = -np.sum(offset * offset, axis=1) / (2 * sigma * sigma)
            weights = np.exp(logits - np.max(logits))
            weights /= weights.sum()
            return np.sum(weights[:, None] * (derivative - GAIN * offset), axis=0)

        solution = solve_ivp(field, (float(left), float(right)), action,
                             method="DOP853", dense_output=True,
                             max_step=DOP_MAX_STEP, **settings)
        if not solution.success or not np.isfinite(solution.y).all():
            raise RuntimeError(f"DOP853 failed in [{left}, {right}]: {solution.message}")
        pieces.append(solution.sol)
        action = solution.y[:, -1].copy()
        nfev += solution.nfev

    def evaluate(times):
        times = np.asarray(times, dtype=np.float64)
        which = np.clip(np.searchsorted(boundaries[1:], times, side="right"), 0, len(pieces)-1)
        result = np.empty((len(times), 2), dtype=np.float64)
        for j, dense in enumerate(pieces):
            use = which == j
            if np.any(use):
                result[use] = dense(times[use]).T
        if not np.isfinite(result).all():
            raise FloatingPointError("Non-finite dense output.")
        return result

    coarse_times = np.unique(np.concatenate([COMMON_TIMES, boundaries]))
    fine_times = np.unique(np.concatenate([FINE_TIMES, boundaries]))
    return dict(common_path=evaluate(COMMON_TIMES),
                coarse_path=evaluate(coarse_times), fine_path=evaluate(fine_times),
                fine_times=fine_times, nfev=int(nfev), pieces=len(pieces))

def path_stats(path, goals):
    if path is None or not np.isfinite(path).all():
        return dict(finite=False, distances=np.full(4, np.inf), worst=np.inf, length=np.inf)
    p0, delta = path[:-1], np.diff(path, axis=0)
    denom = np.sum(delta * delta, axis=1)
    numer = np.sum((goals[None, :, :] - p0[:, None, :]) * delta[:, None, :], axis=-1)
    fraction = np.divide(numer, denom[:, None], out=np.zeros_like(numer), where=denom[:, None] > 0)
    fraction = np.clip(fraction, 0, 1)
    closest = p0[:, None, :] + fraction[..., None] * delta[:, None, :]
    distances = np.linalg.norm(closest - goals[None, :, :], axis=-1).min(axis=0)
    return dict(finite=True, distances=distances, worst=float(distances.max()),
                length=float(np.linalg.norm(delta, axis=1).sum()))

def position_difference(a, b):
    if a is None or b is None or a.shape != b.shape or not np.isfinite(a).all() or not np.isfinite(b).all():
        return np.inf
    return float(np.linalg.norm(a - b, axis=1).max())

def distance_difference(a, b):
    if not a["finite"] or not b["finite"]:
        return np.inf
    return float(np.max(np.abs(a["distances"] - b["distances"])))


## 4. Run and classify each case

- `PASS_STABLE`：两档 DOP853、最高两档 RK4 及两类求解器间达到工作精度，且成功。
- `FAIL_STABLE`：上述检查通过，但仍有目标明显未访问。
- `BOUNDARY`：轨迹达到工作精度，但最远目标距离位于访问阈值 ±0.001 内。
- `UNRESOLVED`：精度检查尚未通过；不把它算成“已确认的结构性失败”。

single 还必须通过精确解检查。同一布局若 single 正对照失败，multi 也标为未解决。
“达到工作精度”不等于严格的数学误差界；本轮只有固定起点，不评价随机起点分布。
表中的 `n_candidates` 是 multi 候选数量，即使该行的方法为 single 也保持同一定义。


In [ ]:
metric_rows, convergence_rows = [], []
completed = set()
plot_cache = {}
single_controls = {}
run_start = time.perf_counter()

def case_info(method, i):
    return dict(method=method, layout_index=int(i), in_sample=bool(i in sample_set),
                gap_bin=str(layout_gap_bins[i]), gap_percent=float(gap_percent[i]),
                n_candidates=len(candidate_ids[i]))

def add_metrics(method, i, solver, stats, elapsed_seconds, nfev=None, error=""):
    for tol in VISIT_TOLERANCES:
        row = dict(**case_info(method, i), solver=solver, tolerance=tol,
                   finite=stats["finite"], success=bool(stats["finite"] and stats["worst"] <= tol),
                   visited_count=int(np.sum(stats["distances"] <= tol)), worst_distance=stats["worst"],
                   path_length=stats["length"], elapsed_seconds=elapsed_seconds, nfev=nfev, error=error)
        row.update({f"target_{j+1}_distance": float(d) for j, d in enumerate(stats["distances"])})
        metric_rows.append(row)

def save_partial():
    if metric_rows:
        atomic_csv(pd.DataFrame(metric_rows), OUTPUT_DIR / "solver_metrics.csv")
    if convergence_rows:
        atomic_csv(pd.DataFrame(convergence_rows), OUTPUT_DIR / "convergence_by_layout.csv")
    manifest["completed_cases"] = len(completed)
    atomic_json(manifest, OUTPUT_DIR / "run_manifest.json")

def run_batch(indices, stage):
    indices = np.asarray(indices, dtype=np.int64)
    for method in METHODS:  # single first, so each multi case has a checked control
        ids = np.asarray([int(i) for i in indices if (method, int(i)) not in completed], dtype=np.int64)
        if not len(ids):
            continue
        print(f"\n[{stage}] {method}: {len(ids)} layouts", flush=True)
        field = make_batch_field(method, ids)
        paths_by_steps, rk_stats = {}, {}
        for steps in RK4_LEVELS:
            print(f"  RK4 {steps} steps: batch started", flush=True)
            start_time = time.perf_counter()
            paths = rk4_batch(field, starts[ids], steps)
            seconds = time.perf_counter() - start_time
            paths_by_steps[steps] = paths
            rk_stats[steps] = [path_stats(path, targets[i]) for i, path in zip(ids, paths)]
            for i, stats in zip(ids, rk_stats[steps]):
                add_metrics(method, int(i), f"rk4_{steps}", stats, seconds / len(ids), nfev=4*steps)
            print(f"  RK4 {steps} done in {seconds:.1f}s", flush=True)

        for b, i0 in enumerate(ids):
            i = int(i0)
            print(f"  DOP853 {b+1}/{len(ids)} | layout {i} | elapsed {time.perf_counter()-run_start:.1f}s",
                  flush=True)
            answers, dop_stats, errors = {}, {}, []
            for solver, settings in DOP_SETTINGS.items():
                start_time = time.perf_counter()
                try:
                    answer = segmented_dop853(method, i, settings)
                    stats = path_stats(answer["fine_path"], targets[i])
                    error = ""
                except (RuntimeError, FloatingPointError, ValueError) as exc:
                    answer, stats, error = None, path_stats(None, targets[i]), str(exc)
                    errors.append(f"{solver}: {error}")
                    print("    ERROR:", error, flush=True)
                answers[solver], dop_stats[solver] = answer, stats
                add_metrics(method, i, solver, stats, time.perf_counter()-start_time,
                            nfev=answer["nfev"] if answer is not None else None, error=error)

            loose, tight = answers["dop853"], answers["dop853_tight"]
            loose_path = loose["common_path"] if loose is not None else None
            tight_path = tight["common_path"] if tight is not None else None
            tight_stats = dop_stats["dop853_tight"]
            coarse_metric_stats = path_stats(tight["coarse_path"] if tight is not None else None, targets[i])
            high, previous = RK4_LEVELS[-1], RK4_LEVELS[-2]
            rk_high = paths_by_steps[high][b]
            dop_position_delta = position_difference(loose_path, tight_path)
            dop_distance_delta = distance_difference(dop_stats["dop853"], tight_stats)
            metric_delta = distance_difference(coarse_metric_stats, tight_stats)
            rk4_position_delta = position_difference(paths_by_steps[previous][b], rk_high[::2])
            cross_position_delta = position_difference(rk_high, tight_path)
            cross_distance_delta = distance_difference(rk_stats[high][b], tight_stats)
            rk4_distance_delta = distance_difference(rk_stats[previous][b], rk_stats[high][b])
            dop_ok = bool(dop_position_delta <= POSITION_TOL and dop_distance_delta <= DISTANCE_TOL
                          and metric_delta <= DISTANCE_TOL)
            rk4_ok = bool(rk4_position_delta <= POSITION_TOL and rk4_distance_delta <= DISTANCE_TOL
                          and cross_position_delta <= POSITION_TOL and cross_distance_delta <= DISTANCE_TOL)

            exact_error = np.nan
            if method == "analytic_single":
                exact_error = position_difference(tight_path, exact_single_path(i, COMMON_TIMES))
                single_controls[i] = bool(exact_error <= SINGLE_EXACT_TOL)
                exact_times = np.unique(np.concatenate([COMMON_TIMES, stored_knots[i]]))
                exact_stats = path_stats(exact_single_path(i, exact_times), targets[i])
                add_metrics(method, i, "single_exact", exact_stats, 0.0, nfev=0)
            single_control_ok = single_controls.get(i, False)
            agreed = bool(dop_ok and rk4_ok and single_control_ok)
            for tol in VISIT_TOLERANCES:
                if not agreed:
                    classification = "UNRESOLVED"
                elif abs(tight_stats["worst"] - tol) <= BOUNDARY_MARGIN:
                    classification = "BOUNDARY"
                elif tight_stats["worst"] <= tol:
                    classification = "PASS_STABLE"
                else:
                    classification = "FAIL_STABLE"
                row = dict(**case_info(method, i), tolerance=tol, classification=classification,
                           numerically_agreed=agreed, dop_agreed=dop_ok, rk4_agreed=rk4_ok,
                           single_control_ok=single_control_ok, single_exact_max_error=exact_error,
                           dop_position_delta=dop_position_delta, dop_distance_delta=dop_distance_delta,
                           metric_grid_distance_delta=metric_delta,
                           rk4_last_position_delta=rk4_position_delta, rk4_last_distance_delta=rk4_distance_delta,
                           rk4_vs_dop_position_delta=cross_position_delta,
                           rk4_vs_dop_distance_delta=cross_distance_delta,
                           dop_tight_success=bool(tight_stats["finite"] and tight_stats["worst"] <= tol),
                           dop_tight_worst_distance=tight_stats["worst"],
                           rk4_coarsest_steps=RK4_LEVELS[0], rk4_highest_steps=high,
                           rk4_coarsest_worst_distance=rk_stats[RK4_LEVELS[0]][b]["worst"],
                           rk4_high_worst_distance=rk_stats[high][b]["worst"], error=" | ".join(errors))
                convergence_rows.append(row)

            if i in PILOT_LAYOUTS:
                cache = {f"rk4_{steps}": paths_by_steps[steps][b].copy() for steps in RK4_LEVELS}
                cache["dop853"], cache["dop853_tight"] = loose_path, tight_path
                plot_cache[(method, i)] = cache
                arrays = {key: value for key, value in cache.items() if value is not None}
                arrays["common_times"] = COMMON_TIMES
                for steps in RK4_LEVELS:
                    arrays[f"rk4_{steps}_times"] = np.linspace(0, 1, steps + 1)
                if tight is not None:
                    arrays["fine_times"], arrays["fine_path"] = tight["fine_times"], tight["fine_path"]
                atomic_npz(OUTPUT_DIR / f"paths_layout_{i}_{method}.npz", **arrays)
            completed.add((method, i))
            if stage == "pilot" or (b+1) % 5 == 0:
                save_partial()
        save_partial()

run_batch(PILOT_LAYOUTS, "pilot")
pilot_frame = pd.DataFrame(convergence_rows)
display(pilot_frame[pilot_frame["tolerance"] == 0.02][[
    "method", "layout_index", "classification", "dop_tight_worst_distance",
    "dop_position_delta", "rk4_vs_dop_position_delta", "single_control_ok",
]])
if (pilot_frame["classification"] == "UNRESOLVED").any():
    print("Some pilot cases remain unresolved. They will stay flagged; the full sample checks their prevalence.")


## 5. Repeat on the same stratified sample

默认自动继续，避免人为挑选结果；已经计算过的典型布局不会重复计算。
即使典型案例有未解决项，也保留这些标签继续检查总体范围。
设置 `RUN_FULL_SAMPLE=False` 可只做典型案例；此时不输出随机样本总体结论。


In [ ]:
if RUN_FULL_SAMPLE:
    run_batch(sample_ids, "sample")
else:
    print("Full sample disabled. These are pilot-only results.")
manifest["status"] = "completed"
manifest["elapsed_seconds"] = time.perf_counter() - run_start
save_partial()

metrics = pd.DataFrame(metric_rows)
convergence = pd.DataFrame(convergence_rows)
scope = "ALL_selected" if RUN_FULL_SAMPLE else "PILOT_only"
chosen_metrics = metrics[metrics["in_sample"]].copy() if RUN_FULL_SAMPLE else metrics.copy()
chosen_convergence = convergence[convergence["in_sample"]].copy() if RUN_FULL_SAMPLE else convergence.copy()
if RUN_FULL_SAMPLE:
    for method in METHODS:
        actual = set(chosen_convergence.loc[chosen_convergence["method"] == method, "layout_index"])
        if actual != sample_set:
            raise RuntimeError("Incomplete sample; inspect partial outputs before interpreting percentages.")

summary_rows = []
for (method, solver, tol), group in chosen_metrics.groupby(["method", "solver", "tolerance"], sort=False):
    for label in [scope, *GAP_LABELS]:
        part = group if label == scope else group[group["gap_bin"] == label]
        if len(part):
            summary_rows.append(dict(method=method, solver=solver, tolerance=tol, gap_bin=label,
                                     n_layouts=len(part), success_percent=100*part["success"].mean(),
                                     mean_visited=part["visited_count"].mean(),
                                     mean_worst_distance=part["worst_distance"].mean(),
                                     nonfinite_layouts=int((~part["finite"]).sum())))
solver_summary = pd.DataFrame(summary_rows)
atomic_csv(solver_summary, OUTPUT_DIR / "solver_summary.csv")

counts = chosen_convergence.groupby(["method", "tolerance", "classification"]).size().unstack(fill_value=0)
counts = counts.reindex(columns=["PASS_STABLE", "FAIL_STABLE", "BOUNDARY", "UNRESOLVED"], fill_value=0)
print("\nClassification counts |", scope)
display(counts)
print("Raw success rates below include unresolved/boundary cases; read together with classification counts.")
display(solver_summary[solver_summary["gap_bin"] == scope][[
    "method", "solver", "tolerance", "n_layouts", "success_percent", "nonfinite_layouts",
]])
display(chosen_convergence[(chosen_convergence["classification"] == "UNRESOLVED") &
                           (chosen_convergence["tolerance"] == 0.02)][[
    "method", "layout_index", "dop_position_delta", "rk4_last_position_delta",
    "rk4_vs_dop_position_delta", "single_control_ok", "error",
]])


## 6. Pilot plots and files to send back

上排：同一解析速度场用不同积分方法得到的轨迹。
下排：RK4 加密后“最难访问的那个目标”的距离；水平线为两档 DOP853 结果和访问阈值。
灰色虚线是候选专家；黑色虚线是更严格的 DOP853 结果。

请发送 `convergence_by_layout.csv`、`solver_summary.csv`、`solver_metrics.csv`，
加上布局 12、20、281、954 的图。如果需要追查个案，目录还保存了对应 `.npz` 轨迹。


In [ ]:
plt.style.use("default")
rk_colors = ["#b8c4d6", "#7594bd", "#3471aa", "#d67222"]
for i in PILOT_LAYOUTS:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    for col, method in enumerate(METHODS):
        ax, distance_ax = axes[0, col], axes[1, col]
        for j in candidate_ids[i]:
            expert = all_routes[i, j]
            ax.plot(expert[:, 0], expert[:, 1], ":", color="0.75", lw=0.9, zorder=1)
        cache = plot_cache[(method, i)]
        for color, steps in zip(rk_colors, RK4_LEVELS):
            path = cache[f"rk4_{steps}"]
            if np.isfinite(path).all():
                ax.plot(path[:, 0], path[:, 1], color=color, lw=1.2, label=f"RK4 {steps}")
        for solver, color, style in [("dop853", "#9b4fac", "-"), ("dop853_tight", "black", "--")]:
            path = cache[solver]
            if path is not None:
                ax.plot(path[:, 0], path[:, 1], color=color, ls=style, lw=1.5, label=solver)
        ax.scatter(*starts[i], marker="*", s=140, color="green", zorder=8)
        ax.scatter(targets[i, :, 0], targets[i, :, 1], marker="s", s=45,
                   color="gold", edgecolors="black", zorder=8)
        for j, point in enumerate(targets[i]):
            ax.add_patch(plt.Circle(point, 0.02, fill=False, color="gray", alpha=0.5))
            ax.annotate(str(j+1), point, xytext=(5, 5), textcoords="offset points")
        row = convergence[(convergence["method"] == method) & (convergence["layout_index"] == i) &
                          (convergence["tolerance"] == 0.02)].iloc[0]
        ax.set_title(f"{method} | {row['classification']} @ 0.02")
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7)
        part = metrics[(metrics["method"] == method) & (metrics["layout_index"] == i) &
                       (metrics["tolerance"] == 0.02)].set_index("solver")
        values = [part.loc[f"rk4_{steps}", "worst_distance"] for steps in RK4_LEVELS]
        distance_ax.plot(RK4_LEVELS, values, "o-", label="RK4 worst target distance")
        for solver, color, style in [("dop853", "#9b4fac", "-"), ("dop853_tight", "black", "--")]:
            value = float(part.loc[solver, "worst_distance"])
            if np.isfinite(value):
                distance_ax.axhline(value, color=color, ls=style, label=solver)
        for tol in VISIT_TOLERANCES:
            distance_ax.axhline(tol, color="red", ls=":", alpha=0.6, label=f"visit tolerance {tol}")
        distance_ax.set_xscale("log", base=2)
        distance_ax.set_xticks(RK4_LEVELS, [str(s) for s in RK4_LEVELS])
        distance_ax.set_xlabel("RK4 integration steps")
        distance_ax.set_ylabel("Maximum of the four minimum target distances")
        distance_ax.grid(alpha=0.2)
        distance_ax.legend(fontsize=7)
    fig.suptitle(f"Layout {i} | fixed start | {len(candidate_ids[i])} candidate routes")
    fig.savefig(OUTPUT_DIR / f"convergence_layout_{i}.png", dpi=170)
    plt.show()
    plt.close(fig)

print("\nComplete. Output directory:", OUTPUT_DIR)
print("Send: convergence_by_layout.csv, solver_summary.csv, solver_metrics.csv, and the four PNGs.")
print("No model training or checkpoint loading occurred.")


## Reading the result

先看 `convergence_by_layout.csv` 的分类，再看成功率。若稳定案例里的 multi 仍漏点，
可以说在本次固定起点、参数和候选规则下，漏点不能全部归因于网络拟合或粗积分。
尚不能仅据此断言所有 SFP、多模态模型或随机初始化都会失败。

`UNRESOLVED` 中，若两档 DOP853 已一致而 RK4 尚未一致，可能只是 RK4 还需加密；
本 notebook 采取保守分类，不强行把 DOP853 当作真值。
`BOUNDARY` 是访问阈值附近的结果，不自动当作已确认的成功或失败。
随机样本是分层各 25 个，不能与原来全量 1000 个布局的比例直接比较。

本轮只研究积分，不更改路线权重、起点、Gaussian 宽度，也不加入路线标签。
如果之后研究“整条轨迹固定路线条件”，那是另一项受控实验，需单独报告。

References:
- SciPy solve_ivp / DOP853: https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html
- Streaming Flow Policy: https://arxiv.org/html/2505.21851v1
